# 🤖 CMF Financial Statements — Hybrid Extractor (Docling + Groq/Llama 4 Scout)

Extracts structured financial data from CMF PDF reports using a **two-pass hybrid approach**:

| Pass | Tool | What it does |
|---|---|---|
| 1 | **Docling** | Fast structural analysis — layout, tables, markdown export |
| 2 | **Groq × Llama 4 Scout Vision** | AI-powered extraction of pages 2–5 → structured JSON |

Results are merged and saved as `.json` alongside each PDF.

---

> ⚠️ **Requires a `GROQ_API_KEY`** — get one free at https://console.groq.com  
> Set it in Cell 3 or in a `.env` file as `GROQ_API_KEY=gsk_...`

> ℹ️ **Model used:** `meta-llama/llama-4-scout-17b-16e-instruct` — Groq's current vision model (supports images + JSON mode). Pixtral and mistral-saba are no longer available on Groq.

## 1. Install Dependencies

In [1]:
!pip install groq pymupdf tenacity docling python-dotenv -q
print("✅ Dependencies installed.")

✅ Dependencies installed.


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
anaconda-cloud-auth 0.1.4 requires pydantic<2.0, but you have pydantic 2.13.1 which is incompatible.


## 2. Imports

In [2]:
import base64
import json
import logging
import os
import random
import sys
import time
from pathlib import Path

import fitz  # PyMuPDF
from groq import Groq, RateLimitError, APIStatusError, APIConnectionError
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
    before_sleep_log,
)
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableStructureOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

try:
    from dotenv import load_dotenv
    load_dotenv()
    print("✅ .env loaded.")
except ImportError:
    print("ℹ️  python-dotenv not found — using environment variables directly.")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)

print("✅ Imports done.")

✅ .env loaded.
✅ Imports done.


## 3. Configuration

> **Set your Groq API key and PDF folder path here.**

In [3]:


# Option B: read from environment / .env file (recommended)
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "YOUR_GROQ_API_KEY")

# Vision model on Groq (supports images + JSON mode)
# meta-llama/llama-4-scout-17b-16e-instruct is Groq's current multimodal model
VISION_MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"

# Folder that contains your CMF PDFs
PDF_FOLDER = r"CMF_SITEX_PDFs"   # ← change this

# DPI for page rendering sent to the model (120 = good balance)
DPI = 120

# ──────────────────────────────────────────────────────────────────────────────

if not GROQ_API_KEY:
    print("❌ GROQ_API_KEY is not set! Add it above or in a .env file.")
    print("   Get a free key at: https://console.groq.com")
else:
    masked = GROQ_API_KEY[:6] + "..." + GROQ_API_KEY[-4:]
    print(f"✅ API key loaded  : {masked}")
    print(f"🤖 Vision model   : {VISION_MODEL}")
    print(f"📁 PDF folder    : {PDF_FOLDER}")

✅ API key loaded  : gsk_CG...r0HZ
🤖 Vision model   : meta-llama/llama-4-scout-17b-16e-instruct
📁 PDF folder    : CMF_SITEX_PDFs


## 4. Groq Client Setup & Connection Test

Uses **`meta-llama/llama-4-scout-17b-16e-instruct`** — Groq's current multimodal model with vision + JSON mode support.

In [4]:
# Initialise the Groq client (used by all extraction functions below)
groq_client = Groq(api_key=GROQ_API_KEY)

def test_groq_connection() -> bool:
    """Send a minimal text ping to verify the API key and connectivity."""
    print(f"🔎 Testing Groq connection with model: {VISION_MODEL}...")
    try:
        resp = groq_client.chat.completions.create(
            model=VISION_MODEL,
            messages=[{"role": "user", "content": "ping"}],
            max_tokens=1,
            temperature=0.0,
        )
        print(f"   ✅ Connected — model confirmed: {resp.model}")
        return True
    except RateLimitError:
        print("   ⚠️ Rate limit hit on ping — model exists, just busy. Continuing...")
        return True
    except APIStatusError as e:
        print(f"   ❌ API error: {e.status_code} — {e.message}")
        return False
    except APIConnectionError as e:
        print(f"   ❌ Connection error: {e}")
        return False

test_groq_connection()

🔎 Testing Groq connection with model: meta-llama/llama-4-scout-17b-16e-instruct...
2026-04-16 14:03:00,014 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
   ✅ Connected — model confirmed: meta-llama/llama-4-scout-17b-16e-instruct


True

## 5. System Prompt

In [5]:
SYSTEM_PROMPT = """Tu es un expert en extraction précise d'états financiers tunisiens/français.
Retourne UNIQUEMENT du JSON valide, sans texte avant ou après, sans balises markdown.

Schéma attendu :
{
  "page_type": "bilan_actif | bilan_passif | cpc | esg | tft | etic | other",
  "title": "titre exact tel qu'imprimé",
  "tables": [
    {
      "caption": "titre du tableau",
      "headers": ["Col1", "Col2"],
      "rows": [{"label": "...", "values": {"Col1": "123456", "Col2": "(78900)"}}]
    }
  ],
  "notes": "texte des notes ou chaîne vide"
}

Règles importantes :
- Recopie les nombres EXACTEMENT comme imprimés (espaces milliers, parenthèses pour négatifs).
- Si la page ne contient pas d'état financier, retourne page_type = "other" et tables = [].
- Ne génère aucun texte en dehors du JSON."""

print("✅ System prompt set.")

✅ System prompt set.


## 6. Core Helper Functions

In [6]:
def page_to_base64(pdf_path: Path, page_number: int, dpi: int = 120) -> str:
    """Render one PDF page to a base64-encoded JPEG string."""
    doc  = fitz.open(str(pdf_path))
    page = doc[page_number]
    mat  = fitz.Matrix(dpi / 72, dpi / 72)
    pix  = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
    doc.close()
    return base64.b64encode(pix.tobytes("jpeg")).decode("utf-8")


def _clean_json_text(raw: str) -> str:
    """Strip markdown code fences the model sometimes adds."""
    raw = raw.strip()
    if raw.startswith("```"):
        lines = raw.split("\n")
        raw = "\n".join(lines[1:])
        if raw.rstrip().endswith("```"):
            raw = raw.rstrip()[:-3]
    return raw.strip()


def _error_page(page_idx: int, message: str) -> dict:
    """Return a placeholder dict for a page that failed to extract."""
    return {
        "page_type": "error",
        "title": "Extraction failed",
        "tables": [],
        "notes": message,
        "_page_number": page_idx + 1,
    }


print("✅ Helper functions defined.")

✅ Helper functions defined.


## 7. Docling Pass — Structural Extraction

In [7]:
def docling_extract(pdf_path: Path) -> dict:
    """Run Docling on the full PDF. Returns markdown text + raw table data."""
    print("   📄 Docling: structural analysis...")
    opts = PdfPipelineOptions(
        do_ocr=False,
        do_table_structure=True,
        table_structure_options=TableStructureOptions(mode="fast"),
    )
    converter = DocumentConverter(
        format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)}
    )
    try:
        result = converter.convert(pdf_path)
        doc    = result.document
        md     = doc.export_to_markdown()
        tables = [
            {
                "caption": getattr(t, "caption", ""),
                "data": [[cell.text for cell in row] for row in t.data.grid],
            }
            for t in doc.tables
        ]
        print(f"   ✅ Docling done ({len(tables)} table(s) found)")
        return {"markdown": md, "tables": tables}
    except Exception as e:
        print(f"   ⚠️ Docling error: {e}")
        return {"markdown": "", "tables": []}


print("✅ docling_extract() ready.")

✅ docling_extract() ready.


## 8. Groq/Llama 4 Scout Vision Pass — Extract Pages 2–5

Each PDF page is rendered as a JPEG and sent to **Llama 4 Scout** via the Groq API.  
This model supports native vision input and JSON mode, making it ideal for structured financial extraction.

In [8]:
@retry(
    stop=stop_after_attempt(8),
    wait=wait_exponential(multiplier=2, min=10, max=90),
    retry=retry_if_exception_type(
        (RateLimitError, APIConnectionError, json.JSONDecodeError)
    ),
    before_sleep=before_sleep_log(logger, logging.WARNING),
    reraise=True,
)
def groq_extract_page(pdf_path: Path, page_number: int, dpi: int = 120) -> dict:
    """
    Render one PDF page → JPEG → send to Mistral Pixtral via Groq.
    Returns parsed JSON dict.

    Retry behaviour:
      - RateLimitError (429)   → retried with exponential back-off
      - APIConnectionError     → retried
      - JSONDecodeError        → retried (model returned malformed JSON)
      - APIStatusError (4xx)   → NOT retried (bad request / auth error)
    """
    b64_image = page_to_base64(pdf_path, page_number, dpi)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{b64_image}"
                    },
                },
                {
                    "type": "text",
                    "text": (
                        f"Ceci est la page {page_number + 1} d'un état financier CMF tunisien. "
                        "Extrais toutes les tables financières et retourne le JSON demandé."
                    ),
                },
            ],
        },
    ]

    response = groq_client.chat.completions.create(
        model=VISION_MODEL,
        messages=messages,
        temperature=0.0,
        max_tokens=4096,
    )

    raw_text = response.choices[0].message.content
    cleaned  = _clean_json_text(raw_text)
    return json.loads(cleaned)   # JSONDecodeError triggers retry


def groq_extract_pages_2_to_5(pdf_path: Path, dpi: int = 120) -> list:
    """Extract pages 2–5 (indices 1–4) from a PDF using Groq/Mistral Vision."""
    print(f"   🔍 Groq [{VISION_MODEL}]: extracting pages 2–5 (Llama 4 Scout Vision)...")

    doc         = fitz.open(str(pdf_path))
    total_pages = len(doc)
    doc.close()

    results = []

    for page_idx in range(1, 5):   # pages 2,3,4,5 → indices 1–4
        if page_idx >= total_pages:
            print(f"      Page {page_idx + 1} — PDF only has {total_pages} page(s), skipping.")
            break

        print(f"      Page {page_idx + 1}/{total_pages} ... ", end="", flush=True)

        try:
            data = groq_extract_page(pdf_path, page_idx, dpi)
            data["_page_number"] = page_idx + 1
            results.append(data)
            print("✅")

        except RateLimitError as e:
            print(f"❌ Rate limit exhausted after retries: {e}")
            results.append(_error_page(page_idx, f"RateLimitError: {e}"))

        except APIStatusError as e:
            print(f"❌ API error {e.status_code}: {e.message}")
            results.append(_error_page(page_idx, f"APIStatusError {e.status_code}: {e.message}"))

        except Exception as e:
            print(f"❌ Unexpected error: {e}")
            results.append(_error_page(page_idx, str(e)))

        # Polite inter-page delay to respect Groq free-tier rate limits
        if page_idx < min(4, total_pages - 1):
            delay = 5.0 + random.uniform(0, 3.0)
            print(f"      ⏱️  Inter-page delay: {delay:.1f}s")
            time.sleep(delay)

    return results


print("✅ Groq/Mistral extraction functions ready.")

✅ Groq/Mistral extraction functions ready.


## 9. Merge Results

In [9]:
def merge_results(docling_data: dict, groq_pages: list) -> dict:
    """Combine Docling structural output with Groq/Llama 4 Scout page extractions."""
    output = {
        "source":               "hybrid_docling_groq_llama4scout",
        "model":                VISION_MODEL,
        "pages_analyzed":       len(groq_pages),
        "financial_statements": {},
        "docling_markdown":     docling_data.get("markdown", ""),
    }

    by_type: dict[str, list] = {}
    for page in groq_pages:
        pt = page.get("page_type", "other")
        by_type.setdefault(pt, []).append(page)

    for stmt_type, pages in by_type.items():
        tables = [
            {"page": p.get("_page_number"), **t}
            for p in pages
            for t in p.get("tables", [])
        ]
        output["financial_statements"][stmt_type] = {
            "title":  pages[0].get("title", stmt_type),
            "tables": tables,
            "notes":  " | ".join(
                p.get("notes", "") for p in pages if p.get("notes")
            ),
        }

    return output


print("✅ merge_results() ready.")

✅ merge_results() ready.


## 10. Run — Process All PDFs in Folder

Each PDF gets a `_extract_pages2-5.json` output file saved next to it.  
Already-processed PDFs are **skipped automatically**.

In [10]:
folder = Path(PDF_FOLDER)

if not folder.exists() or not folder.is_dir():
    print(f"❌ Folder not found: {folder}")
else:
    pdf_files = sorted(list(folder.glob("*.pdf")) + list(folder.glob("*.PDF")))

    if not pdf_files:
        print("❌ No PDF files found in the folder.")
    else:
        print(f"📂 Found {len(pdf_files)} PDF(s). Starting hybrid extraction...\n")

        succeeded, skipped, failed = [], [], []

        for pdf_file in pdf_files:
            print(f"{'=' * 60}")
            print(f"🚀 Processing → {pdf_file.name}")
            print(f"{'=' * 60}")

            output_file = folder / f"{pdf_file.stem}_extract_pages2-5.json"

            if output_file.exists():
                print(f"⏭️  Already processed → {output_file.name}\n")
                skipped.append(pdf_file.name)
                continue

            try:
                # ── Pass 1: Docling ────────────────────────────────────────
                docling_data = docling_extract(pdf_file)

                # ── Pass 2: Groq/Mistral Vision ────────────────────────────
                groq_pages   = groq_extract_pages_2_to_5(pdf_file, dpi=DPI)

                # ── Merge & save ───────────────────────────────────────────
                final = merge_results(docling_data, groq_pages)
                output_file.write_text(
                    json.dumps(final, ensure_ascii=False, indent=2),
                    encoding="utf-8",
                )

                detected = list(final["financial_statements"].keys()) or ["None"]
                print(f"✅ Saved → {output_file.name}")
                print(f"   Statements detected: {detected}\n")
                succeeded.append(pdf_file.name)

            except Exception as e:
                print(f"❌ Critical error with {pdf_file.name}: {e}\n")
                failed.append((pdf_file.name, str(e)))

        # ── Final summary ──────────────────────────────────────────────────
        print(f"\n{'=' * 60}")
        print("🎉 Batch complete!")
        print(f"   ✅ Succeeded : {len(succeeded)}")
        print(f"   ⏭️  Skipped   : {len(skipped)}")
        print(f"   ❌ Failed    : {len(failed)}")
        if failed:
            print("\n   Failed files:")
            for name, err in failed:
                print(f"     • {name}: {err}")

📂 Found 20 PDF(s). Starting hybrid extraction...

🚀 Processing → sitex-efd-3112-2020.pdf
   📄 Docling: structural analysis...
2026-04-16 14:03:25,377 [INFO] detected formats: [<InputFormat.PDF: 'pdf'>]
2026-04-16 14:03:25,881 [INFO] Going to convert document batch...
2026-04-16 14:03:25,883 [INFO] Initializing pipeline for StandardPdfPipeline with options hash d18651824ca6fe56cf3ccb20424ac2c8
2026-04-16 14:03:26,027 [INFO] Loading plugin 'docling_defaults'
2026-04-16 14:03:26,044 [INFO] Registered picture descriptions: ['picture_description_vlm_engine', 'vlm', 'api']
2026-04-16 14:03:26,212 [INFO] Loading plugin 'docling_defaults'
2026-04-16 14:03:26,275 [INFO] Registered ocr engines: ['auto', 'easyocr', 'kserve_v2_ocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2026-04-16 14:03:26,493 [INFO] Loading plugin 'docling_defaults'
2026-04-16 14:03:26,520 [INFO] Registered layout engines: ['layout_object_detection', 'docling_layout_default', 'docling_experimental_table_crops_layout']
2

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

2026-04-16 14:03:28,254 [INFO] Loading plugin 'docling_defaults'
2026-04-16 14:03:28,269 [INFO] Registered table structure engines: ['docling_tableformer', 'docling_tableformer_v2']
2026-04-16 14:03:28,458 [INFO] HTTP Request: GET https://huggingface.co/api/models/docling-project/docling-models/revision/v2.3.0 "HTTP/1.1 200 OK"
2026-04-16 14:03:28,815 [INFO] Accelerator device: 'cpu'
2026-04-16 14:03:29,849 [INFO] Processing document sitex-efd-3112-2020.pdf
2026-04-16 14:03:34,337 [ERROR] Stage preprocess failed for run 1, pages [8]: std::bad_alloc
2026-04-16 14:03:34,869 [ERROR] Stage preprocess failed for run 1, pages [9]: std::bad_alloc
2026-04-16 14:03:35,013 [ERROR] Stage preprocess failed for run 1, pages [10]: std::bad_alloc
2026-04-16 14:03:35,314 [ERROR] Stage preprocess failed for run 1, pages [11]: std::bad_alloc
2026-04-16 14:03:35,561 [ERROR] Stage preprocess failed for run 1, pages [12]: std::bad_alloc
2026-04-16 14:03:35,727 [ERROR] Stage preprocess failed for run 1, pag

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

2026-04-16 14:05:23,215 [INFO] HTTP Request: GET https://huggingface.co/api/models/docling-project/docling-models/revision/v2.3.0 "HTTP/1.1 200 OK"
2026-04-16 14:05:23,243 [INFO] Accelerator device: 'cpu'
2026-04-16 14:05:24,157 [INFO] Processing document sitex_efd311215.pdf
2026-04-16 14:05:26,725 [ERROR] Stage preprocess failed for run 1, pages [7]: std::bad_alloc
2026-04-16 14:05:27,128 [ERROR] Stage preprocess failed for run 1, pages [8]: std::bad_alloc
2026-04-16 14:05:27,288 [ERROR] Stage preprocess failed for run 1, pages [9]: std::bad_alloc
2026-04-16 14:05:27,435 [ERROR] Stage preprocess failed for run 1, pages [10]: std::bad_alloc
2026-04-16 14:05:27,703 [ERROR] Stage preprocess failed for run 1, pages [11]: std::bad_alloc
2026-04-16 14:05:27,920 [ERROR] Stage preprocess failed for run 1, pages [12]: std::bad_alloc
2026-04-16 14:05:28,111 [ERROR] Stage preprocess failed for run 1, pages [13]: std::bad_alloc
2026-04-16 14:05:28,246 [ERROR] Stage preprocess failed for run 1, pa

...

2026-04-16 14:14:19,181 [INFO] HTTP Request: GET https://huggingface.co/api/models/docling-project/docling-models/revision/v2.3.0 "HTTP/1.1 200 OK"
2026-04-16 14:14:19,195 [INFO] Accelerator device: 'cpu'
2026-04-16 14:14:19,609 [INFO] Processing document sitex_efd311222.pdf
2026-04-16 14:14:20,803 [ERROR] Stage preprocess failed for run 1, pages [6]: std::bad_alloc
2026-04-16 14:14:20,878 [ERROR] Stage preprocess failed for run 1, pages [7]: std::bad_alloc
2026-04-16 14:14:20,940 [ERROR] Stage preprocess failed for run 1, pages [8]: std::bad_alloc
2026-04-16 14:14:21,011 [ERROR] Stage preprocess failed for run 1, pages [9]: std::bad_alloc
2026-04-16 14:14:21,120 [ERROR] Stage preprocess failed for run 1, pages [10]: std::bad_alloc
2026-04-16 14:14:21,426 [ERROR] Stage preprocess failed for run 1, pages [11]: std::bad_alloc
2026-04-16 14:14:21,488 [ERROR] Stage preprocess failed for run 1, pages [12]: std::bad_alloc
2026-04-16 14:14:21,564 [ERROR] Stage preprocess failed for run 1, pag

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

2026-04-16 14:15:46,283 [INFO] HTTP Request: GET https://huggingface.co/api/models/docling-project/docling-models/revision/v2.3.0 "HTTP/1.1 200 OK"
2026-04-16 14:15:46,298 [INFO] Accelerator device: 'cpu'
2026-04-16 14:15:46,950 [INFO] Processing document sitex_efd311223.pdf
2026-04-16 14:15:47,692 [ERROR] Stage preprocess failed for run 1, pages [6]: std::bad_alloc
2026-04-16 14:15:47,758 [ERROR] Stage preprocess failed for run 1, pages [7]: std::bad_alloc
2026-04-16 14:15:47,817 [ERROR] Stage preprocess failed for run 1, pages [8]: std::bad_alloc
2026-04-16 14:15:47,880 [ERROR] Stage preprocess failed for run 1, pages [9]: std::bad_alloc
2026-04-16 14:15:47,937 [ERROR] Stage preprocess failed for run 1, pages [10]: std::bad_alloc
2026-04-16 14:15:48,010 [ERROR] Stage preprocess failed for run 1, pages [11]: std::bad_alloc
2026-04-16 14:15:48,109 [ERROR] Stage preprocess failed for run 1, pages [12]: std::bad_alloc
2026-04-16 14:15:48,173 [ERROR] Stage preprocess failed for run 1, pag

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

2026-04-16 14:17:04,952 [INFO] HTTP Request: GET https://huggingface.co/api/models/docling-project/docling-models/revision/v2.3.0 "HTTP/1.1 200 OK"
2026-04-16 14:17:04,976 [INFO] Accelerator device: 'cpu'
2026-04-16 14:17:05,665 [INFO] Processing document sitex_efd311224.pdf
2026-04-16 14:17:11,126 [ERROR] Stage preprocess failed for run 1, pages [6]: std::bad_alloc
2026-04-16 14:17:11,259 [ERROR] Stage preprocess failed for run 1, pages [7]: std::bad_alloc
2026-04-16 14:17:11,380 [ERROR] Stage preprocess failed for run 1, pages [8]: std::bad_alloc
2026-04-16 14:17:11,449 [ERROR] Stage preprocess failed for run 1, pages [9]: std::bad_alloc
2026-04-16 14:17:11,543 [ERROR] Stage preprocess failed for run 1, pages [10]: std::bad_alloc
2026-04-16 14:17:11,644 [ERROR] Stage preprocess failed for run 1, pages [11]: std::bad_alloc
2026-04-16 14:17:11,728 [ERROR] Stage preprocess failed for run 1, pages [12]: std::bad_alloc
2026-04-16 14:17:11,812 [ERROR] Stage preprocess failed for run 1, pag

## 11. (Optional) Inspect a Single Output JSON

In [11]:
# Set this to any output JSON file you want to preview
INSPECT_FILE = ""   # e.g. r"C:/path/to/report_extract_pages2-5.json"

if INSPECT_FILE:
    with open(INSPECT_FILE, encoding="utf-8") as f:
        data = json.load(f)

    print(f"Source       : {data.get('source')}")
    print(f"Model        : {data.get('model')}")
    print(f"Pages parsed : {data.get('pages_analyzed')}")
    print(f"Statements   : {list(data['financial_statements'].keys())}")
    print()

    for stmt, content in data["financial_statements"].items():
        print(f"── {stmt} : {content['title']} ──")
        for tbl in content["tables"]:
            print(f"   Table (page {tbl.get('page')}): {tbl.get('caption', 'no caption')}")
            print(f"   Headers : {tbl.get('headers', [])}")
            print(f"   Rows    : {len(tbl.get('rows', []))}")
        print()
else:
    print("Set INSPECT_FILE to a JSON path to preview its contents.")

Set INSPECT_FILE to a JSON path to preview its contents.


---
## ✅ Done!

Each PDF now has a `_extract_pages2-5.json` file containing:

```json
{
  "source": "hybrid_docling_groq_llama4scout",
  "model": "meta-llama/llama-4-scout-17b-16e-instruct",
  "pages_analyzed": 4,
  "financial_statements": {
    "bilan_actif":  { "title": "...", "tables": [...] },
    "bilan_passif": { "title": "...", "tables": [...] }
  },
  "docling_markdown": "..."
}
```

**Tips:**
- **Rate limits** — Groq free tier allows ~30 req/min. The inter-page delay handles this automatically.
- **More pages** — change `range(1, 5)` in Cell 8 to e.g. `range(1, 10)` to scan more pages.
- **Higher quality** — increase `DPI` in Cell 3 (e.g. `150` or `200`) for denser tables.
- **Get your free Groq API key** at https://console.groq.com